In [1]:
!pip install --upgrade "mlflow>=3.1"
!pip install pandas
!pip install scikit-learn
!pip install boto3

In [2]:
from datetime import datetime
import pandas as pd
import s3fs
import mlflow
import mlflow.sklearn
from mlflow.tracking import MlflowClient
from mlflow.models import infer_signature
import json
import shutil

/home/ec2-user/anaconda3/envs/python3/lib/python3.12/site-packages/pydantic/_internal/_fields.py:132: UserWarning: Field "model_name" in PromptModelConfig has conflict with protected namespace "model_".

You may be able to resolve this warning by setting `model_config['protected_namespaces'] = ()`.
  warnings.warn(


In [3]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, classification_report, f1_score
from sklearn.model_selection import train_test_split

In [25]:
# =======================================================
# TODO MIEMBRO DEL EQUIPO: MODIFICAR SOLO ESTA VARIABLE
# =======================================================
# 1. La MEJOR carpeta que ganó en la Fase A (Ej: "Exp01_Solo_Lema")
WINNING_SPACY_FOLDER = "Exp04_Solo_Elongacion"

# 2. Configuración de Codificación a evaluar
TIPO_VECTORIZADOR = "TFIDF" # Opciones: "BOW" (CountVectorizer) o "TFIDF"
USE_BIGRAMS = True          # False = (1,1) Unigramas | True = (1,2) Bigramas

EXPERIMENT_FOLDER = f"Codificacion_{TIPO_VECTORIZADOR}_{'Bigrama' if USE_BIGRAMS else 'Unigrama'}"

# ---------- S3 ----------
S3_BUCKET = "parcial-pln"
TRAIN_PATH = f"parcial-pln/data/processed/Spacy/{WINNING_SPACY_FOLDER}/train_parquet"

# Usamos 'models_experiments' para NO disparar la Lambda a Producción
DEST_BUCKET = "s3://parcial-pln/data/models_experiments/FaseB_Codificacion" 
DATASET_VERSION = WINNING_SPACY_FOLDER

# ---------- MLflow ----------
MLFLOW_TRACKING_URI = "http://ec2-52-21-111-46.compute-1.amazonaws.com:5000"
MLFLOW_ARTIFACT_BUCKET = "s3://artifacts-mlflow-pln"
EXPERIMENT_NAME = "Sentimientos403_FaseB"
REGISTERED_MODEL_NAME = "Sentimientos403_Codificacion"
MODEL_TYPE = "Codificacion_NB"
TEAM = "NPL"

# ---------- Modelo ----------
TEXT_COLUMN = "ablation_text"
TARGET_COLUMN = "label"
MAX_FEATURES = 5000
AUTOR_EXPERIMENTO = "Daniel Varela"

In [26]:
fs = s3fs.S3FileSystem()

def load_parquet_from_s3(prefix: str) -> pd.DataFrame:
    files = fs.ls(prefix)
    df_list = [pd.read_parquet(f"s3://{file}", filesystem=fs) for file in files]
    return pd.concat(df_list, ignore_index=True)

print(f"Cargando dataset ganador de la Fase A: {WINNING_SPACY_FOLDER}...")
full_train_df = load_parquet_from_s3(TRAIN_PATH)

Cargando dataset ganador de la Fase A: Exp04_Solo_Elongacion...


In [27]:
X = full_train_df[TEXT_COLUMN].fillna("") 
y = full_train_df[TARGET_COLUMN]

print("Dividiendo en Entrenamiento (80%) y Validación (20%)...")
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

Dividiendo en Entrenamiento (80%) y Validación (20%)...


In [28]:
# Configurar rango de N-gramas
ngram_range = (1, 2) if USE_BIGRAMS else (1, 1)

# Seleccionar vectorizador dinámicamente
if TIPO_VECTORIZADOR == "TFIDF":
    vectorizer = TfidfVectorizer(max_features=MAX_FEATURES, ngram_range=ngram_range)
elif TIPO_VECTORIZADOR == "BOW":
    vectorizer = CountVectorizer(max_features=MAX_FEATURES, ngram_range=ngram_range)
else:
    raise ValueError("TIPO_VECTORIZADOR debe ser 'TFIDF' o 'BOW'")

pipeline = Pipeline([
    ("vectorizer", vectorizer),
    ("model", MultinomialNB())
])

print(f"Pipeline configurado: {TIPO_VECTORIZADOR} con N-gramas {ngram_range}")

Pipeline configurado: TFIDF con N-gramas (1, 2)


In [29]:
mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
experiment = mlflow.get_experiment_by_name(EXPERIMENT_NAME)

if experiment is None:
    artifact_location = f"{MLFLOW_ARTIFACT_BUCKET}/{EXPERIMENT_NAME}/"
    print(f"Creando experimento '{EXPERIMENT_NAME}' con artefactos en {artifact_location}")
    mlflow.create_experiment(name=EXPERIMENT_NAME, artifact_location=artifact_location)
else:
    print(f"Utilizando experimento existente '{EXPERIMENT_NAME}'")

mlflow.set_experiment(EXPERIMENT_NAME)
client = MlflowClient()

Utilizando experimento existente 'Sentimientos403_FaseB'


In [30]:
with mlflow.start_run(run_name=EXPERIMENT_FOLDER):

    mlflow.set_tag("Autor", AUTOR_EXPERIMENTO)
    mlflow.set_tag("dataset_version", DATASET_VERSION)
    mlflow.set_tag("model_type", MODEL_TYPE)
    mlflow.set_tag("team", TEAM)

    print(f"Entrenando modelo: {EXPERIMENT_FOLDER}...")
    pipeline.fit(X_train, y_train)

    y_pred = pipeline.predict(X_val)

    acc = accuracy_score(y_val, y_pred)
    f1 = f1_score(y_val, y_pred, average="macro")

    print(f"Accuracy: {acc:.4f}")
    print(f"F1 Macro: {f1:.4f}")

    # Registramos la configuración específica de codificación
    mlflow.log_params({
        "model_type": "MultinomialNB",
        "vectorizer": TIPO_VECTORIZADOR,
        "use_bigrams": USE_BIGRAMS,
        "max_features": MAX_FEATURES,
        "spacy_base": WINNING_SPACY_FOLDER
    })

    mlflow.log_metrics({
        "accuracy": acc,
        "f1_macro": f1
    })

    signature = infer_signature(X_train, pipeline.predict(X_train))

    mlflow.sklearn.log_model(
        pipeline,
        artifact_path="CodificacionModel",
        signature=signature,
        input_example=X_train.iloc[:3].to_frame(),
        registered_model_name=REGISTERED_MODEL_NAME
    )

print("Modelo registrado correctamente.")

Entrenando modelo: Codificacion_TFIDF_Bigrama...
Accuracy: 0.7775
F1 Macro: 0.7775


2026/03/06 02:56:30 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/03/06 02:56:31 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
Registered model 'Sentimientos403_Codificacion' already exists. Creating a new version of this model...
2026/03/06 02:56:34 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: Sentimientos403_Codificacion, version 4
Created version '4' of model 'Sentimientos403_Codificacion'.


🏃 View run Codificacion_TFIDF_Bigrama at: http://ec2-52-21-111-46.compute-1.amazonaws.com:5000/#/experiments/10/runs/096d1fa4ca47419bb90b6058779a3bc1
🧪 View experiment at: http://ec2-52-21-111-46.compute-1.amazonaws.com:5000/#/experiments/10
Modelo registrado correctamente.


In [31]:
local_tmp = "tmp_champion_model"

########################### MODEL CARD ######################
model_card = {
    "model_name": EXPERIMENT_FOLDER,
    "description": f"Modelo de evaluación de codificación usando {TIPO_VECTORIZADOR} y {'Bigramas' if USE_BIGRAMS else 'Unigramas'}.",
    "author": AUTOR_EXPERIMENTO,
    "dataset_version": DATASET_VERSION,
    "training_data_path": TRAIN_PATH,
    "version": str(datetime.now()),
    "performance": {
        "validation_accuracy": round(acc, 4),
        "validation_f1_macro": round(f1, 4)
    },
    "intended_use": "Encontrar la mejor representación vectorial de los textos.",
    "limitations": ["Dependiente de las features extraídas por spaCy"]
}

################################# MLFLOW CHECKIN #############################
versions = client.search_model_versions(f"name='{REGISTERED_MODEL_NAME}'")
latest_version = max(int(v.version) for v in versions)

print(f"Nueva versión registrada: {latest_version}")

try:
    champion_info = client.get_model_version_by_alias(REGISTERED_MODEL_NAME, "champion")
    champion_version = int(champion_info.version)
    champion_run = client.get_run(champion_info.run_id)
    champion_f1 = champion_run.data.metrics.get("f1_macro", 0)
    print(f"Champion actual: v{champion_version} | F1={champion_f1:.4f}")
except Exception:
    champion_f1 = -1
    print("No existe champion aún en la Fase B.")

print(f"📊 Nuevo modelo F1: {f1:.4f}")

if f1 > champion_f1:
    print("¡Nuevo Champion de Codificación! Promoviendo...")
    
    # 1. Promover en MLflow
    client.set_registered_model_alias(name=REGISTERED_MODEL_NAME, alias="champion", version=latest_version)
    client.set_model_version_tag(name=REGISTERED_MODEL_NAME, version=latest_version, key="status", value="champion")

    # 2. Guardar Model Card en S3
    with fs.open(f"{DEST_BUCKET}/model_card.json", "w") as f:
        f.write(json.dumps(model_card, indent=4))
    
    # 3. Sincronizar Modelo a la carpeta de experimentos Fase B
    mlflow.artifacts.download_artifacts(artifact_uri=f"models:/{REGISTERED_MODEL_NAME}@champion", dst_path=local_tmp)
    fs.put(local_tmp, DEST_BUCKET, recursive=True)
    shutil.rmtree(local_tmp)
    print("Artefactos sincronizados exitosamente en S3 (models_experiments/FaseB_Codificacion).")

else:
    print("El champion actual sigue siendo mejor. No se sobrescriben archivos en S3.")

Nueva versión registrada: 4
Champion actual: v1 | F1=0.7748
📊 Nuevo modelo F1: 0.7775
¡Nuevo Champion de Codificación! Promoviendo...


Artefactos sincronizados exitosamente en S3 (models_experiments/FaseB_Codificacion).
